In [1]:
import numpy as np
from math import erf, sqrt

# --- primitive payoff functions ---
def research(x):
    return float(200_000 * (np.log(1 + x) / np.log(101)))

def scale(x):
    return float(7 * x / 100)

def payoff_res_sc(budget_res, budget_sc):
    return research(budget_res) * scale(budget_sc)

def max_payoff_res_sc(budget):
    """Optimal split of `budget` between research and scale. Returns (r, s, payoff)."""
    X = np.linspace(0, budget, 1000)
    payoffs = [(float(x), float(budget - x), payoff_res_sc(float(x), float(budget - x))) for x in X]
    payoffs.sort(key=lambda t: t[2], reverse=True)
    return payoffs[0]

def normal_cdf(x, mean, sd):
    return 0.5 * (1 + erf((x - mean) / (sd * np.sqrt(2))))

# Lookup: integer budget (0..100) -> optimal (research + scale) payoff
optimal_lookup = {b: max_payoff_res_sc(b)[2] for b in range(101)}

# --- best-response sweep over opponent-belief (m, s) grid ---
MEAN_LIST = list(np.linspace(20, 80, 10))
SD_LIST   = list(np.linspace(10, 30, 5))
best_responses = []                               # (m, s, v*, payoff*)

for m in MEAN_LIST:
    for s in SD_LIST:
        best_v, best_payoff = 0, 0.0
        for v in range(101):                      # your own speed investment
            b = 100 - v                           # remaining budget for res+scale
            rank = normal_cdf(v, m, s)            # P(opponent's speed <= v)
            multiplier = 0.1 + 0.8 * rank
            payoff = optimal_lookup[b] * multiplier
            if payoff > best_payoff:
                best_v, best_payoff = v, payoff
        best_responses.append((float(m), float(s), best_v, best_payoff))

best_responses

[(20.0, 10.0, 32, 361299.8501072529),
 (20.0, 15.0, 33, 326900.93442902004),
 (20.0, 20.0, 32, 304352.40132216696),
 (20.0, 25.0, 30, 290140.74086956563),
 (20.0, 30.0, 27, 281891.9186408574),
 (26.666666666666668, 10.0, 38, 315230.9741923907),
 (26.666666666666668, 15.0, 38, 284728.28841707425),
 (26.666666666666668, 20.0, 37, 265749.46099555225),
 (26.666666666666668, 25.0, 34, 254561.76778250356),
 (26.666666666666668, 30.0, 31, 248912.7321467742),
 (33.333333333333336, 10.0, 44, 271066.85915850673),
 (33.333333333333336, 15.0, 44, 244718.59501672487),
 (33.333333333333336, 20.0, 42, 229347.82232369814),
 (33.333333333333336, 25.0, 39, 221225.94413584494),
 (33.333333333333336, 30.0, 35, 218036.6733774724),
 (40.0, 10.0, 49, 229016.09031984935),
 (40.0, 15.0, 49, 207063.94752000575),
 (40.0, 20.0, 46, 195298.69021237604),
 (40.0, 25.0, 43, 190180.80303458142),
 (40.0, 30.0, 39, 189356.00080767716),
 (46.66666666666667, 10.0, 55, 189530.83304313198),
 (46.66666666666667, 15.0, 54, 17

In [2]:
# =============================================================
# (1) Symmetric pure-strategy fixed point:
#     find m* such that best-response to opponents ~ N(m*, s) equals m*.
#     Damped best-response iteration for each s.
# =============================================================

def best_response(m, s):
    """Best integer speed v in [0,100] against opponents ~ N(m, s). Returns (v*, payoff)."""
    best_v, best_payoff = 0, 0.0
    for v in range(101):
        b = 100 - v
        rank = normal_cdf(v, m, s)
        mult = 0.1 + 0.8 * rank
        payoff = optimal_lookup[b] * mult
        if payoff > best_payoff:
            best_v, best_payoff = v, payoff
    return best_v, best_payoff

def find_fixed_point(s, m_init=50.0, alpha=0.5, tol=0.5, max_iter=200):
    """Iterate m <- (1-alpha)*m + alpha*BR(m, s) until |BR - m| < tol."""
    m = float(m_init)
    for i in range(max_iter):
        v_star, payoff = best_response(m, s)
        if abs(v_star - m) < tol:
            return {"s": s, "m_star": m, "v_star": v_star,
                    "payoff": payoff, "iters": i + 1, "converged": True}
        m = (1 - alpha) * m + alpha * v_star
    return {"s": s, "m_star": m, "v_star": v_star,
            "payoff": payoff, "iters": max_iter, "converged": False}

S_GRID = [5, 10, 15, 20, 25, 30]
fixed_points = [find_fixed_point(s) for s in S_GRID]

print("Symmetric pure-strategy fixed points (opponents ~ N(m*, s)):")
print(f"{'s':>4} {'m*':>7} {'v*':>4} {'payoff':>12} {'iters':>6} {'conv':>5}")
for fp in fixed_points:
    print(f"{fp['s']:>4} {fp['m_star']:>7.2f} {fp['v_star']:>4d} "
          f"{fp['payoff']:>12.1f} {fp['iters']:>6d} {str(fp['converged']):>5}")

# =============================================================
# (4) Robust / minimax: pick v that maximises worst-case payoff
#     over a plausible grid of opponent beliefs (m, s).
# =============================================================

M_GRID_ROBUST = np.linspace(20, 70, 11)
S_GRID_ROBUST = np.linspace(10, 30, 5)

worst_case = []                                # (v, min_payoff, (m_worst, s_worst))
for v in range(101):
    b = 100 - v
    min_payoff = float("inf")
    worst_ms = None
    for m in M_GRID_ROBUST:
        for s in S_GRID_ROBUST:
            rank = normal_cdf(v, m, s)
            mult = 0.1 + 0.8 * rank
            payoff = optimal_lookup[b] * mult
            if payoff < min_payoff:
                min_payoff = payoff
                worst_ms = (float(m), float(s))
    worst_case.append((v, min_payoff, worst_ms))

v_robust, robust_payoff, worst_ms_at_opt = max(worst_case, key=lambda x: x[1])
rs_split = max_payoff_res_sc(100 - v_robust)

print("\nRobust (max-min) best response:")
print(f"  v* (speed)         = {v_robust}")
print(f"  worst-case payoff  = {robust_payoff:,.1f}")
print(f"  worst-case belief  = N(m={worst_ms_at_opt[0]:.1f}, s={worst_ms_at_opt[1]:.1f})")
print(f"  optimal split of remaining {100 - v_robust}% budget:")
print(f"     research = {rs_split[0]:.2f}%,  scale = {rs_split[1]:.2f}%")

Symmetric pure-strategy fixed points (opponents ~ N(m*, s)):
   s      m*   v*       payoff  iters  conv
   5   41.23   45     285457.0    200 False
  10   70.68   72      77742.0    200 False
  15   65.60   66      88867.4     11  True
  20   55.61   56     126068.2      7  True
  25   48.38   48     154662.7      4  True
  30   38.36   38     196223.4     10  True

Robust (max-min) best response:
  v* (speed)         = 0
  worst-case payoff  = 74,233.6
  worst-case belief  = N(m=70.0, s=10.0)
  optimal split of remaining 100% budget:
     research = 23.12%,  scale = 76.88%


In [3]:
# Scenario:
#   - my speed investment        = 44%
#   - opponents' speed ~ N(m=33.33, s=15)
#   - remaining 56% split optimally between research and scale
#   - full 50,000 XIREC budget is used

BUDGET_TOTAL = 50_000
my_speed = 44
m_opp, s_opp = 33.33, 15.0

res_sc_budget = 100 - my_speed                       # 56%
budget_rs, budget_sc, res_sc_payoff = max_payoff_res_sc(res_sc_budget)

rank        = normal_cdf(my_speed, m_opp, s_opp)     # P(opponent speed < 44)
multiplier  = 0.1 + 0.8 * rank
gross_pnl   = res_sc_payoff * multiplier
net_pnl     = gross_pnl - BUDGET_TOTAL

print(f"Speed investment      : {my_speed}%")
print(f"Research investment   : {budget_rs:.2f}%")
print(f"Scale investment      : {budget_sc:.2f}%")
print(f"Opponent speed belief : N(m={m_opp}, s={s_opp})")
print(f"Rank percentile       : {rank:.4f}")
print(f"Speed multiplier      : {multiplier:.4f}")
print(f"Research x Scale      : {res_sc_payoff:,.2f}")
print(f"Gross PnL (R x S x V) : {gross_pnl:,.2f}")
print(f"Budget used           : {BUDGET_TOTAL:,}")
print(f"Net PnL               : {net_pnl:,.2f}")

Speed investment      : 44%
Research investment   : 14.29%
Scale investment      : 41.71%
Opponent speed belief : N(m=33.33, s=15.0)
Rank percentile       : 0.7616
Speed multiplier      : 0.7092
Research x Scale      : 345,065.88
Gross PnL (R x S x V) : 244,737.60
Budget used           : 50,000
Net PnL               : 194,737.60


In [4]:
# Integer-percentage version of the previous scenario.
#   - speed = 44%
#   - remaining 56% split between integer research r and integer scale s (r + s = 56)
#   - opponents' speed ~ N(33.33, 15)

my_speed = 44
res_sc_budget = 100 - my_speed                          # 56

best_r, best_s, best_rs_payoff = 0, 0, -float("inf")
for r in range(res_sc_budget + 1):                      # r = 0..56
    s = res_sc_budget - r
    p = research(r) * scale(s)
    if p > best_rs_payoff:
        best_r, best_s, best_rs_payoff = r, s, p

m_opp, s_opp = 33.33, 15.0
rank       = normal_cdf(my_speed, m_opp, s_opp)
multiplier = 0.1 + 0.8 * rank
gross_pnl  = best_rs_payoff * multiplier
net_pnl    = gross_pnl - BUDGET_TOTAL

print(f"Speed investment      : {my_speed}%")
print(f"Research investment   : {best_r}%   (integer)")
print(f"Scale investment      : {best_s}%   (integer)")
print(f"Research x Scale      : {best_rs_payoff:,.2f}")
print(f"Speed multiplier      : {multiplier:.4f}")
print(f"Gross PnL (R x S x V) : {gross_pnl:,.2f}")
print(f"Budget used           : {BUDGET_TOTAL:,}")
print(f"Net PnL               : {net_pnl:,.2f}")

Speed investment      : 44%
Research investment   : 14%   (integer)
Scale investment      : 42%   (integer)
Research x Scale      : 345,025.34
Speed multiplier      : 0.7092
Gross PnL (R x S x V) : 244,708.85
Budget used           : 50,000
Net PnL               : 194,708.85
